In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

train_csv_path = "../input/plant-pathology-2021-fgvc8/train.csv"
sample_sub_path = "../input/plant-pathology-2021-fgvc8/sample_submission.csv"
train_images_dir = "../input/plant-pathology-2021-fgvc8/train_images"
test_images_dir = "../input/plant-pathology-2021-fgvc8/test_images"
output_path = "submission.csv"

# Choose device: GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # speed up cuDNN on fixed image size




In [2]:
train_df = pd.read_csv(train_csv_path)

mlb = MultiLabelBinarizer()
train_labels = mlb.fit_transform(train_df["labels"].apply(lambda x: x.split()))




In [3]:
transform = T.Compose(
    [
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

base_model = models.mobilenet_v3_small(pretrained=True).to(device)
base_model.eval()
feature_extractor = torch.nn.Sequential(
    base_model.features, base_model.avgpool, torch.nn.Flatten()
).to(device)
feature_extractor.eval()


class ImageDataset(Dataset):
    def __init__(self, img_ids, img_dir):
        self.img_ids = list(img_ids)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = os.path.join(self.img_dir, img_id)
        img = Image.open(img_path).convert("RGB")
        return transform(img)


def extract_features_batch(img_ids, img_dir, batch_size=64):
    dataset = ImageDataset(img_ids, img_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=os.cpu_count(),
        pin_memory=True,
    )
    all_feats = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Batches"):
            batch = batch.to(device, non_blocking=True)
            feats = feature_extractor(batch)
            all_feats.append(feats.cpu().numpy())
    return np.concatenate(all_feats, axis=0)


print("Extracting training features...")
train_feats = extract_features_batch(train_df["image"], train_images_dir)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:00<00:00, 83.5MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 32 worker processes in total. Our 

Extracting training features...


Batches: 100%|██████████| 233/233 [06:27<00:00,  1.66s/it]


In [4]:
print("Training logistic regression classifier...")
clf = OneVsRestClassifier(LogisticRegression(solver="lbfgs", max_iter=500, n_jobs=-1))
clf.fit(train_feats, train_labels)




Training logistic regression classifier...


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

OneVsRestClassifier(estimator=LogisticRegression(max_iter=500, n_jobs=-1))

In [5]:
sub_df = pd.read_csv(sample_sub_path)

print("Extracting test features...")
test_feats = extract_features_batch(sub_df["image"], test_images_dir)

print("Predicting labels for test set...")
pred_probs = clf.predict_proba(test_feats)  # list of arrays per class
pred_binary = (pred_probs >= 0.5).astype(int)

pred_labels = mlb.inverse_transform(pred_binary)
sub_df["labels"] = [" ".join(lbls) if lbls else "healthy" for lbls in pred_labels]



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Extracting test features...


Batches: 100%|██████████| 59/59 [01:43<00:00,  1.75s/it]

Predicting labels for test set...


In [6]:
sub_df.to_csv(output_path, index=False)
print(f"Submission saved to {output_path}")

Submission saved to submission.csv
